In [1]:
%%bash
cat << 'EOF' > super_pi.c
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char **argv) {
    int accuracy = 20; 
    if (argc > 1) accuracy = atoi(argv[1]);
    long long iterations = 20000000LL * accuracy; 
    
    double a = 1.0, pi = 0.0;
    for (long long i = 0; i < iterations; i++) {
        pi += a / (2 * i + 1);
        a = -a;
    }

    // Read exact scheduler metrics from the kernel before exiting
    FILE *f = fopen("/proc/self/schedstat", "r");
    if (f) {
        unsigned long long exec_ns, wait_ns, slices;
        // schedstat format: exec_runtime wait_runtime run_times
        if (fscanf(f, "%llu %llu %llu", &exec_ns, &wait_ns, &slices) == 3) {
            // Print out execution time, wait latency, and context switches
            printf("%llu,%llu,%llu\n", exec_ns, wait_ns, slices);
        }
        fclose(f);
    }
    return 0;
}
EOF

gcc -O3 super_pi.c -o super_pi
chmod +x super_pi
echo "Self-reporting Super PI benchmark compiled successfully."

Self-reporting Super PI benchmark compiled successfully.


In [1]:
import subprocess
import time
import numpy as np
import pandas as pd
from IPython.display import display
# -----------------------
def run_workload(total_runs, num_procs, accuracy, csv_filename):
    print(f"Executing {total_runs} benchmark runs.")
    print(f"Workload: {num_procs} processes, accuracy {accuracy}")
    print("-" * 105)
    
    all_runs_data = []
    
    for run in range(1, total_runs + 1):
        processes = []
        total_start = time.time()
        
        # taskset -c 0 pins all processes to CPU 0 to force scheduler contention
        cmd = ["taskset", "-c", "0", "./super_pi", accuracy]
    
        # Spawn processes and capture their stdout
        for _ in range(num_procs):
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, text=True)
            processes.append(p)
    
        turnaround_times = []
        wait_latencies = []
        context_switches = []
    
        # Wait for completion and parse the kernel metrics
        for p in processes:
            out, _ = p.communicate()
            turnaround_times.append(time.time() - total_start)
            
            if out.strip():
                try:
                    # Parse the CSV output from our C program
                    exec_ns, wait_ns, slices = map(int, out.strip().split(','))
                    wait_latencies.append(wait_ns / 1e9)  # Convert nanoseconds to seconds
                    context_switches.append(slices)
                except ValueError:
                    pass
    
        # --- Macro-Metrics ---
        std_dev = np.std(turnaround_times)                 # Fairness
        avg_turnaround = np.mean(turnaround_times)         # Responsiveness
        max_turnaround = np.max(turnaround_times)          # System completion time
        throughput = num_procs / max_turnaround            # Tasks per second
        
        # --- Micro-Metrics ---
        avg_latency = np.mean(wait_latencies) if wait_latencies else 0       # Avg time spent starving
        avg_switches = np.mean(context_switches) if context_switches else 0  # Avg times scheduled
        total_switches = np.sum(context_switches) if context_switches else 0 # Total context switches
    
        run_label = f"Run {run:02d}" + (" (Warm-up)" if run == 1 else "")
        
        all_runs_data.append({
            "Run": run_label,
            "Std_Dev_s": round(std_dev, 4),
            "Avg_Turn_s": round(avg_turnaround, 4),
            "Throughput_tps": round(throughput, 4),
            "Avg_Latency_s": round(avg_latency, 4),
            "Avg_Context_Switches": round(avg_switches, 1),
            "Total_Context_Switches": total_switches
        })
    
        print(f"{run_label:<18} | StdDev: {std_dev:.4f}s | Latency: {avg_latency:.4f}s | Switches: {total_switches} | Throughput: {throughput:.2f} tps")
    
    print("-" * 105)
    print(f"Testing complete. Saving data to {csv_filename}...")
    
    # Export to CSV and display a clean table in Jupyter
    df = pd.DataFrame(all_runs_data)
    df.to_csv(csv_filename, index=False)
    display(df)

In [5]:
# --- Test Parameters ---
num_procs = 1000     # Number of concurrent processes
accuracy = "95"    # Workload intensity
total_runs = 11    # 1 warm-up + 10 recorded runs

# IMPORTANT: Change this filename when you test the eBPF scheduler!
csv_filename = "eevdf_baseline_metrics.csv" 
run_workload(total_runs, num_procs, accuracy, csv_filename)

Executing 11 benchmark runs.
Workload: 1000 processes, accuracy 95
---------------------------------------------------------------------------------------------------------
Run 01 (Warm-up)   | StdDev: 0.0128s | Latency: 0.0037s | Switches: 2241 | Throughput: 713.24 tps
Run 02             | StdDev: 0.0127s | Latency: 0.0019s | Switches: 2217 | Throughput: 862.98 tps
Run 03             | StdDev: 0.0129s | Latency: 0.0019s | Switches: 2276 | Throughput: 866.32 tps
Run 04             | StdDev: 0.0142s | Latency: 0.0020s | Switches: 2237 | Throughput: 853.95 tps
Run 05             | StdDev: 0.0131s | Latency: 0.0014s | Switches: 2242 | Throughput: 859.42 tps
Run 06             | StdDev: 0.0133s | Latency: 0.0021s | Switches: 2244 | Throughput: 861.38 tps
Run 07             | StdDev: 0.0126s | Latency: 0.0020s | Switches: 2224 | Throughput: 866.90 tps
Run 08             | StdDev: 0.0129s | Latency: 0.0016s | Switches: 2219 | Throughput: 858.47 tps
Run 09             | StdDev: 0.0133s | Late

,Run,Std_Dev_s,Avg_Turn_s,Throughput_tps,Avg_Latency_s,Avg_Context_Switches,Total_Context_Switches
0,Run 01 (Warm-up),0.0128,1.3831,713.2450,0.0037,2.2,2241
1,Run 02,0.0127,1.1391,862.9758,0.0019,2.2,2217
2,Run 03,0.0129,1.1347,866.3197,0.0019,2.3,2276
3,Run 04,0.0142,1.1501,853.9482,0.0020,2.2,2237
4,Run 05,0.0131,1.1431,859.4244,0.0014,2.2,2242
5,Run 06,0.0133,1.1404,861.3797,0.0021,2.2,2244
6,Run 07,0.0126,1.1345,866.8966,0.0020,2.2,2224
7,Run 08,0.0129,1.1457,858.4719,0.0016,2.2,2219
8,Run 09,0.0133,1.1300,869.8376,0.0023,2.3,2262
9,Run 10,0.0133,1.1352,865.5808,0.0025,2.3,2251
